# Module 15: Geospatial Integrity & Observability

In this exercise, we move beyond simple "valid polygons" to build a multi-dimensional validation pipeline. We categorize data integrity into **Three Pillars**:

1. **Geometric Integrity**: Topology (OGC validity) and Projection Detectors.
2. **Metadata & Schema Integrity**: Schema drift and mandatory tag compliance.
3. **Contextual & Operational Integrity**: Detecting staleness and coverage drops.

We will implement a **Quarantine Router** pattern to ensure only high-quality data reaches the user.

In [8]:
import os
import time
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Polygon, Point, MultiPolygon
import duckdb
import lonboard
from datetime import datetime, timedelta

os.makedirs("data", exist_ok=True)
print("Setup complete.")

Setup complete.


## Part 1: Generating the "Multi-Failure" Dataset

We will create a synthetic dataset of 5 features, each representing a different integrity failure mode.

**Note on Schema Drift**: In the cell below, we use a GeoJSON format for the initial "Dirty" ingestion. Why? Because strictly typed formats like Parquet/Arrow are *already* integrity guards—they will crash (as you may have seen) if you try to save mixed types into a single column. GeoJSON is more "relaxed," making it a common source of schema drift in the real world.

In [9]:
# 1. Clean Feature (Reference)
box_clean = Polygon([(0, 0), (1, 0), (1, 1), (0, 1), (0, 0)])

# 2. Broken Topology (Self-intersecting bow-tie)
box_broken = Polygon([(2, 0), (3, 1), (2, 1), (3, 0), (2, 0)])

# 3. Projection Trap (Labeled as EPSG:4326 but coordinates are in meters/Web Mercator)
box_mercator = Polygon([(500000, 500000), (500001, 500000), (500001, 500001), (500000, 500001), (500000, 500000)])

data = [
    {"id": 1, "geometry": box_clean, "priority": 1, "source": "survey_a", "timestamp": datetime.now()},
    {"id": 2, "geometry": box_broken, "priority": 2, "source": "survey_b", "timestamp": datetime.now()},
    {"id": 3, "geometry": box_mercator, "priority": 1, "source": "legacy_db", "timestamp": datetime.now()},
    {"id": 4, "geometry": box_clean, "priority": "HIGH", "source": "manual_entry", "timestamp": datetime.now()}, # Schema Drift: string priority
    {"id": 5, "geometry": box_clean, "priority": 1, "source": None, "timestamp": datetime.now() - timedelta(days=10)} # Metadata Gap & Staleness
]

gdf = gpd.GeoDataFrame(data, crs="EPSG:4326")

# To allow 'Schema Drift' (mixed types) to exist for this exercise, 
# we'll bypass the strict Parquet check for the 'Dirty' source and use GeoJSON.
dirty_path = "data/dirty_dataset.geojson"
gdf.to_file(dirty_path, driver='GeoJSON')
print(f"Created dirty GeoJSON dataset with {len(gdf)} records.")

Created dirty GeoJSON dataset with 5 records.


## Part 2: The Three Pillars Validation Engine

We implement checkers for each pillar. Note how we load the GeoJSON back into a GeoDataFrame to begin our QC.

In [10]:
gdf_dirty = gpd.read_file("data/dirty_dataset.geojson")

def validate_record(row):
    errors = []
    
    # Pillar 1: Geometric Integrity
    if row.geometry is None or not row.geometry.is_valid:
        errors.append("GEOM_INVALID")
    
    # Projection Detection: Are coordinates outside WGS84 bounds (-180, 180)?
    bounds = row.geometry.bounds
    if any(abs(b) > 180 for b in bounds):
        errors.append("PROJECTION_MISMATCH")

    # Pillar 2: Metadata & Schema Integrity
    if pd.isna(row['source']):
        errors.append("METADATA_GAP")
    
    # Check if priority is an integer. GeoJSON might have stringified it.
    try:
        int(row['priority'])
    except (ValueError, TypeError):
        errors.append("SCHEMA_DRIFT")

    # Pillar 3: Contextual & Operational Integrity
    # Convert string timestamp back to datetime if necessary
    ts = pd.to_datetime(row['timestamp'])
    age = datetime.now() - ts.to_pydatetime()
    if age.days > 7:
        errors.append("DATA_STALENESS")
        
    return ",".join(errors) if errors else "VALID"

gdf_dirty['status'] = gdf_dirty.apply(validate_record, axis=1)
print("Validation complete.")
gdf_dirty[['id', 'status']]

Validation complete.


,id,status
0,1,VALID
1,2,GEOM_INVALID
2,3,PROJECTION_MISMATCH
3,4,SCHEMA_DRIFT
4,5,"METADATA_GAP,DATA_STALENESS"


## Part 3: The Quarantine Router

We programmatically split the data into Production (Clean) and Quarantine (Dirty).

**Self-Healing**: Now that we've filtered out the "Dirty" data, our `to_parquet` call for `PROD` will succeed because the schema is finally consistent!

In [11]:
prod_gdf = gdf_dirty[gdf_dirty['status'] == "VALID"].copy()
quarantine_gdf = gdf_dirty[gdf_dirty['status'] != "VALID"].copy()

print(f"Routing Results:")
print(f"- PROD: {len(prod_gdf)} records")
print(f"- QUARANTINE: {len(quarantine_gdf)} records")

# Fix types for production
prod_gdf['priority'] = prod_gdf['priority'].astype(int)

prod_gdf.to_parquet("data/prod_clean.parquet")
quarantine_gdf.to_parquet("data/quarantine_review.parquet") # Status column makes this safe to save
print("Saved PROD and QUARANTINE buckets successfully.")

Routing Results:
- PROD: 1 records
- QUARANTINE: 4 records
Saved PROD and QUARANTINE buckets successfully.


## Part 4: Visual Failure Analysis

We visualize the quarantine bucket. 

**Note on Lonboard Validation**: You may have noticed that Lonboard (Deck.gl) *also* acts as an integrity guard. If you try to visualize a 'Projection Trap' (e.g., longitude = 500,000), Lonboard will crash with a `ValueError`. 

To build a robust **Data Steward Dashboard**, we must filter out values that can't even be mapped!

In [12]:
# Filter out the projection trap features so Lonboard can render the rest of the quarantine bucket
vis_gdf = quarantine_gdf[~quarantine_gdf['status'].str.contains("PROJECTION_MISMATCH")].copy()
outlier_count = len(quarantine_gdf) - len(vis_gdf)

print(f"Visualizing {len(vis_gdf)} manageable failures.")
print(f"Note: {outlier_count} projection outliers were removed from visualization to prevent renderer crash.")

m = lonboard.viz(vis_gdf)
m

Visualizing 3 manageable failures.
Note: 1 projection outliers were removed from visualization to prevent renderer crash.
